In [4]:
from pathlib import Path
import datetime
import re
import torch
import json

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM

In [6]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt, max_length=4000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

Loading checkpoint shards: 100%|██████████| 3/3 [00:16<00:00,  5.45s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


In [7]:
dream_title = ResponseSchema(
    name="dream-title",
    description="A concise meaningful title the discribe the dream. Mainly consists of three or less words."
)
dream_date = ResponseSchema(
    name="dream_date",
    description="The date (in DD-MM-YYYY format) in which the dream happened. {date}"
)
dream_desc = ResponseSchema(
    name="dream_description",
    description="A exact replica of the user inputed dream word for word."
)
dream_symbols = ResponseSchema(
    name="dream_symbols",
    description="A list of all the possible symbolism cotained in the dream."
)
dream_vibes = ResponseSchema(
    name="dream_vibes",
    description="A short list of words that discribe the main feeling (vibe) of the dream."
)

response_schemas = [dream_title,
                    dream_date,
                    dream_desc,
                    dream_symbols, 
                    dream_vibes]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [8]:
dream_journal_template_prompt = """
You are an expert dream journaler and analyst that extracts dream details based on the user's input.



Extract all qualifications as follows:

dream title
dream date {date}
dream description, including details about what happened, how it felt, and any related ideas
dream symbols
dream vibes, describing the main feeling of the dream


Respond ONLY in Markdown format as follows:
{format_instructions}

Example Input:
"
I had a dream were I was runing away from a mirror
"

Expected output (im markdown):
"
# Dream Title
The Mirror

**Dream Date:** 27-07-2026

## Dream

## Dream Description
I had a dream were I was runing away from a mirror

## Dream Symbols
- Mirror
- Avoidance

## Dream Vibes
- Fear
- Avoidance
"

Now extract from the following input:
"{user_input}"
"""

In [9]:
def ask_question(query):
    prompt = f"""You are a helpful assistant. Use the following context to answer the question. Question: {query}"""
    
    result = generate_text(prompt)
    return result.strip()

In [10]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [11]:
date = datetime.date.today().strftime("%d-%m-%Y")

In [13]:
user_input = "I had a dream where I was eating then suddenly I turned into a 100m tall gaint!"

prompt = PromptTemplate(
    template=dream_journal_template_prompt,
    input_variables=["user_input", "format_instructions", "date"]
).format(user_input=user_input, format_instructions=format_instructions, date=date)

In [14]:
answer = ask_question(prompt)
print("\n Answer:", answer.split("Answer:")[-1], "\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



 Answer: You are a helpful assistant. Use the following context to answer the question. Question: 
You are an expert dream journaler and analyst that extracts dream details based on the user's input.



Extract all qualifications as follows:

dream title
dream date 27-07-2026
dream description, including details about what happened, how it felt, and any related ideas
dream symbols
dream vibes, describing the main feeling of the dream


Respond ONLY in Markdown format as follows:
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"dream-title": string  // A concise meaningful title the discribe the dream. Mainly consists of three or less words.
	"dream_date": string  // The date (in DD-MM-YYYY format) in which the dream happened. {date}
	"dream_description": string  // A exact replica of the user inputed dream word for word.
	"dream_symbols": string  // A list of all the possible symbolism 

In [19]:
json_text = extract_json_block(answer)
print(json_text)

```json
{
	"dream-title": "Giant Transformation",
	"dream_date": "27-07-2026",
	"dream_description": "I had a dream where I was eating then suddenly I turned into a 100m tall giant!",
	"dream_symbols": [
		"Eating",
		"Transformation",
		"Giant"
	],
	"dream_vibes": [
		"Surprise",
		"Awe"
	]
}
```


In [23]:
output_data = output_parser.parse(json_text)
print(output_data)

{'dream-title': 'Giant Transformation', 'dream_date': '27-07-2026', 'dream_description': 'I had a dream where I was eating then suddenly I turned into a 100m tall giant!', 'dream_symbols': ['Eating', 'Transformation', 'Giant'], 'dream_vibes': ['Surprise', 'Awe']}


In [24]:
# Ensure database directory exists
db_dir = Path("database")
db_dir.mkdir(parents=True, exist_ok=True)

# Remove code fences if the model wrapped the JSON output

def clean_json_text(text):
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if len(lines) >= 2:
            text = "\n".join(lines[1:-1])
    return text

# Try to parse the model output into JSON and create a clean JSON file
try:
    clean_text = clean_json_text(json_text)
    obj = json.loads(clean_text)
    json_content = json.dumps(obj, indent=2, ensure_ascii=False)
    data = obj
except Exception:
    data = output_data if hasattr(output_data, "keys") else {}
    json_content = json.dumps(data, indent=2, ensure_ascii=False)

# Build a clean Markdown journal entry using the parsed data
def build_markdown_entry(data):
    title = data.get("dream-title", "Untitled Dream")
    dream_date_value = data.get("dream_date", date)
    description = data.get("dream_description", "")
    symbols = data.get("dream_symbols", [])
    vibes = data.get("dream_vibes", [])

    if not isinstance(symbols, list):
        symbols = [symbols]
    if not isinstance(vibes, list):
        vibes = [vibes]

    lines = [
        f"# {title}",
        "",
        f"**Dream Date:** {dream_date_value}",
        "",
        "## Dream",
        "",
        "## Dream Description",
        description,
        "",
        "## Dream Symbols",
    ]
    lines.extend(f"- {symbol}" for symbol in symbols)
    lines.extend(["", "## Dream Vibes"])
    lines.extend(f"- {vibe}" for vibe in vibes)
    return "\n".join(lines)

md_content = build_markdown_entry(data)

# Write JSON file named after the date, with safe filename characters
safe_date = re.sub(r"[^A-Za-z0-9._-]+", "_", str(date))
json_path = db_dir / f"{safe_date}.json"
with open(json_path, "w", encoding="utf-8") as f:
    f.write(json_content)

# Write Markdown file as a plain journal entry
md_path = db_dir / f"{safe_date}.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write(md_content)

print(f"Saved: {json_path} and {md_path}")

Saved: database\27-07-2026.json and database\27-07-2026.md
